In [1]:
import warnings
warnings.filterwarnings('ignore')
import os, re, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from numba import njit, prange
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

# Data 
DATA_START = '2000-01-01'
END_DATE = '2026-01-01'
BENCHMARK = '^SP500TR'
RF_TICKER = '^IRX' # 13-week T-bill, annualized percent
TRADING_DAYS = 252

# Rolling windows
FORMATION_MONTHS = 12 # form pairs on the first 12 months
TRADING_MONTHS = 6 # trade them on the next 6 months
STEP_MONTHS = 6 # then step forward 6 months
FIRST_FORMATION = '2000-01-01'
LAST_FORMATION = '2025-12-31'

# Pair selection 
K_OPTION = [2,4,7] # candidate cluster amounts, chosen by CVI for each window (Silhouette is the tie-breaker)
N_PAIRS = 20 # pairs traded per method

# Trading rule: following the Gatev et al. paper (2006)
ENTRY_SD = 2.0   # Open when the spread moves 2 formation-period standard deviations
LAG_DAYS = 1    # Trade the day after the signal
COST_LEVELS_BPS = [0, 5, 10, 20]

# Significance testing
N_BOOT, BLOCK_DAYS, BOOT_SEED = 5000, 20, 42

# Data Screening carried over from the first study 
KNOWN_EXCLUDE = ['FNMA', 'FMCC', 'EP', 'CCU', 'CB']

# Models 
METHODS = ['MAN', 'DTW', 'OCP', 'GATEV']
LABELS = {'MAN': 'Clustered Manhattan', 'DTW': 'Clustered DTW', 'OCP': 'Clustered OCP',
          'GATEV': 'Gatev Distance', 'Naive': 'Naive Buy-and-Hold'}
COLORS = {'MAN': '#9467bd', 'DTW': '#1f77b4', 'OCP': '#ff7f0e', 'GATEV': '#2ca02c', 'Naive': '#7f7f7f'}
PLANNED_COMPARISONS = [('MAN', 'GATEV'), ('DTW', 'MAN'), ('OCP', 'DTW')]

# Files
TAG = 'rolling_sp500'
CACHE_DIR = f'cache_{TAG}'
os.makedirs(CACHE_DIR, exist_ok=True)

print(f'Rolling design: form {FORMATION_MONTHS} months, trade {TRADING_MONTHS} months, step {STEP_MONTHS} months')
print(f'Models: {[LABELS[m] for m in METHODS]}')
print(f'Planned comparisons: {[f"{LABELS[a]} vs {LABELS[b]}" for a, b in PLANNED_COMPARISONS]}')
print(f'Cache folder: {CACHE_DIR}')

Rolling design: form 12 months, trade 6 months, step 6 months
Models: ['Clustered Manhattan', 'Clustered DTW', 'Clustered OCP', 'Gatev Distance']
Planned comparisons: ['Clustered Manhattan vs Gatev Distance', 'Clustered DTW vs Clustered Manhattan', 'Clustered OCP vs Clustered DTW']
Cache folder: cache_rolling_sp500


In [2]:
hist_url = ('https://raw.githubusercontent.com/fja05680/sp500/master/'
            'S%26P%20500%20Historical%20Components%20%26%20Changes%20(Updated).csv')
hist = pd.read_csv(hist_url)
hist['date'] = pd.to_datetime(hist['date'])
hist = hist.sort_values('date').reset_index(drop=True)
hist.to_csv(f'sp500_membership_{TAG}.csv', index=False)

suffix_pattern = re.compile(r'-\d{6}$')
def clean(t):
    return suffix_pattern.sub('', t.strip()).replace('.', '-')

def members_on(date):
    row = hist[hist['date'] <= date].iloc[-1]
    tickers = {clean(t) for t in row['tickers'].split(',')}
    return sorted(tickers - set(KNOWN_EXCLUDE))

windows = []
form_start = pd.Timestamp(FIRST_FORMATION)
last_end = pd.Timestamp(LAST_FORMATION)
while True:
    form_end  = form_start + pd.DateOffset(months=FORMATION_MONTHS) - pd.Timedelta(days=1)
    trade_start = form_end + pd.Timedelta(days=1)
    trade_end = trade_start + pd.DateOffset(months = TRADING_MONTHS) - pd.Timedelta(days=1)
    if trade_end > last_end:
        break
    windows.append({'window': len(windows) + 1,
                    'form_start': form_start, 'form_end': form_end,
                    'trade_start': trade_start, 'trade_end': trade_end,
                    'members': members_on(form_start)})
    form_start = form_start + pd.DateOffset(months=STEP_MONTHS)

window_df = pd.DataFrame(windows)
window_df['n_members'] = window_df['members'].apply(len)
print(window_df[['window', 'form_start', 'form_end', 'trade_start', 'trade_end', 'n_members']].to_string(index=False))

all_tickers = sorted(set().union(*window_df['members']))
print(f'\n{len(windows)} windows; {len(all_tickers)} distinct tickers across all windows')

with open(f'{CACHE_DIR}/windows.json','w') as f:
    json.dump([{k: (str(v.date()) if isinstance(v, pd.Timestamp) else v) for k, v in w.items()}
              for w in windows], f, indent=1)

 window form_start   form_end trade_start  trade_end  n_members
      1 2000-01-01 2000-12-31  2001-01-01 2001-06-30        487
      2 2000-07-01 2001-06-30  2001-07-01 2001-12-31        487
      3 2001-01-01 2001-12-31  2002-01-01 2002-06-30        487
      4 2001-07-01 2002-06-30  2002-07-01 2002-12-31        490
      5 2002-01-01 2002-12-31  2003-01-01 2003-06-30        491
      6 2002-07-01 2003-06-30  2003-07-01 2003-12-31        490
      7 2003-01-01 2003-12-31  2004-01-01 2004-06-30        489
      8 2003-07-01 2004-06-30  2004-07-01 2004-12-31        489
      9 2004-01-01 2004-12-31  2005-01-01 2005-06-30        489
     10 2004-07-01 2005-06-30  2005-07-01 2005-12-31        490
     11 2005-01-01 2005-12-31  2006-01-01 2006-06-30        490
     12 2005-07-01 2006-06-30  2006-07-01 2006-12-31        491
     13 2006-01-01 2006-12-31  2007-01-01 2007-06-30        492
     14 2006-07-01 2007-06-30  2007-07-01 2007-12-31        492
     15 2007-01-01 2007-12-31  2008-01-0

In [3]:
# Daily prices, benchmark and T-bill: load frozen files if they exist, otherwise download and save
PRICE_FILE = f'daily_prices_{TAG}.csv.gz'
BENCH_FILE = f'bench_daily_{TAG}.csv'
RF_FILE = f'rf_daily_{TAG}.csv'

def to_close_frame(raw):
    close = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw[['Close']]
    return close if isinstance(close, pd.DataFrame) else close.to_frame()

if os.path.exists(PRICE_FILE):
    prices = pd.read_csv(PRICE_FILE, index_col=0, parse_dates=True)
    print(f'Loaded frozen prices from {PRICE_FILE}')
else: 
    raw = yf.download(all_tickers, start=DATA_START, end=END_DATE, interval='1d',
                      auto_adjust=True, progress=True, threads=True)
    prices = to_close_frame(raw).dropna(how='all')   #drop days with no data
    prices = prices.dropna(axis=1, how='all')
    prices.to_csv(PRICE_FILE, float_format='%.6f')
    print(f'Downloaded and saved prices to {PRICE_FILE}')

missing = sorted(set(all_tickers) - set(prices.columns))
print(f'Tickers with data: {prices.shape[1]} of {len(all_tickers)} ({len(missing)} with none)')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()} ({len(prices):,} trading days)')
print('Sanity check, large current stock present:',
      {t: t in prices.columns for t in ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ']})   

# Benchmark (context only)
if os.path.exists(BENCH_FILE):
    bench = pd.read_csv(BENCH_FILE, index_col=0, parse_dates=True).iloc[:, 0]
else:
    b = to_close_frame(yf.download(BENCHMARK, start=DATA_START, end=END_DATE, interval='1d',
                                   auto_adjust=True, progress=False))
    bench = b.iloc[:,0].reindex(prices.index)
    print(f'Benchmark missing days before filling: {int(bench.isna().sum())}')
    bench = bench.ffill()
    bench.to_csv(BENCH_FILE)

# Daily risk-free rate, known at the start of each day
if os.path.exists(RF_FILE):
    rf_daily = pd.read_csv(RF_FILE, index_col=0, parse_dates=True).iloc[:, 0]
else:
    i = to_close_frame(yf.download(RF_TICKER, start=DATA_START, end=END_DATE, interval='1d',
                                   auto_adjust=False, progress=False))
    irx = i.iloc[:, 0].reindex(prices.index).ffill().bfill()
    rf_daily = (np.log1p(irx / 100) / TRADING_DAYS).shift(1).bfill()
    rf_daily.to_csv(RF_FILE)
    print(f'T-bill mean annual yield: {irx.mean():.2f}%')

$JOY: possibly delisted; no timezone found
$CTLT: possibly delisted; no timezone found      ]
[                       0%                       ]  2 of 1047 completed$ATVI: possibly delisted; no timezone found
$WLL: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AWE"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: OWENQ"}}}
$AWE: possibly delisted; no timezone found
$MOLX: possibly delisted; no price data found  (1d 2000-01-01 -> 2026-01-01) (Yahoo error = "No data found, symbol may be delisted")
[                       1%                       ]  6 of 1047 completed$OWENQ: possibly delisted; no timezone found
[                       1%                       ]  6 of 1047 completed$BMS: possibly delisted; no price data found  (1d 2000-01-01 -> 2026-01-01) (Yahoo error = "No data found, symbol may be delisted")

Downloaded and saved prices to daily_prices_rolling_sp500.csv.gz
Tickers with data: 647 of 1047 (400 with none)
Date range: 2000-01-03 to 2025-12-31 (6,539 trading days)
Sanity check, large current stock present: {'AAPL': True, 'MSFT': True, 'JPM': True, 'XOM': True, 'JNJ': True}


In [6]:
orig_url = ('https://raw.githubusercontent.com/fja05680/sp500/master/'
            'S%26P%20500%20Historical%20Components%20%26%20Changes.csv')
ORIG_FILE = f'sp500_membership_original_{TAG}.csv'
if os.path.exists(ORIG_FILE):
    orig = pd.read_csv(ORIG_FILE, parse_dates=['date'])
else:
    orig = pd.read_csv(orig_url)
    orig['date'] = pd.to_datetime(orig['date'])
    orig = orig.sort_values('date').reset_index(drop=True)
    orig.to_csv(ORIG_FILE, index=False)

def reused_in_window(form_start):
    if form_start > orig['date'].max():
        return set()                                   # no delisting markers after 2019
    row = orig[orig['date'] <= form_start].iloc[-1]
    raw = [t.strip() for t in row['tickers'].split(',')]
    plain = {clean(t) for t in raw if not suffix_pattern.search(t)}
    reused = set()
    for t in raw:
        if suffix_pattern.search(t):
            c = clean(t)
            d = pd.Timestamp(f'{t[-6:-2]}-{t[-2:]}-01') + pd.offsets.MonthEnd(0)
            if c not in plain and c in prices.columns:
                if prices.loc[d + pd.Timedelta(days=31):, c].notna().any():
                    reused.add(c)
    return reused

# 2. Per-window universe, including the reused-ticker rule
def window_universe(form_start, form_end):
    row = hist[hist['date'] <= form_start].iloc[-1]
    members = {clean(t) for t in row['tickers'].split(',')} - set(KNOWN_EXCLUDE) - otc
    reused = reused_in_window(form_start)
    form = prices.loc[form_start:form_end]
    with_data = [t for t in members if t in form.columns]
    complete = [t for t in with_data if form[t].notna().all()]
    usable = [t for t in complete if t not in reused]
    return sorted(usable), len(members), len(with_data), len(complete) - len(usable)

summary = []
for w in windows:
    usable, n_mem, n_data, n_reused = window_universe(w['form_start'], w['form_end'])
    w['universe'] = usable
    summary.append({'window': w['window'], 'form_start': w['form_start'].date(),
                    'members': n_mem, 'with_data': n_data,
                    'reused_removed': n_reused, 'usable': len(usable)})
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv(f'universe_by_window_{TAG}.csv', index=False)
with open(f'{CACHE_DIR}/universes.json', 'w') as f:
    json.dump({str(w['window']): w['universe'] for w in windows}, f)
print(f"\nUsable stocks per window: min {summary_df['usable'].min()}, "
      f"median {int(summary_df['usable'].median())}, max {summary_df['usable'].max()}")

 window form_start  members  with_data  reused_removed  usable
      1 2000-01-01      486        275               3     221
      2 2000-07-01      486        277               2     224
      3 2001-01-01      487        285               2     232
      4 2001-07-01      490        289               2     236
      5 2002-01-01      491        288               3     237
      6 2002-07-01      490        292               4     239
      7 2003-01-01      489        297               3     245
      8 2003-07-01      489        299               3     247
      9 2004-01-01      489        300               4     248
     10 2004-07-01      490        299               4     251
     11 2005-01-01      490        298               4     251
     12 2005-07-01      491        299               4     252
     13 2006-01-01      492        306               4     263
     14 2006-07-01      492        312               4     269
     15 2007-01-01      492        314               5 

In [12]:
# Formation-year inputs and the Manhattan distance (no warping)
def formation_paths(w):
    px = prices.loc[w['form_start']:w['form_end'], w['universe']]
    X = (px/px.iloc[0] - 1.0).values.astype(np.float64)  # cumulative returns from day 1, starts at 0
    return X, list(px.columns)

@njit(parallel=True, cache=True)
def manhattan_matrix(X):
    T, n = X.shape
    D = np.zeros((n,n))
    for i in prange(n):
        for j in range(i + 1, n):
            s = 0.0
            for t in range(T):
                s += abs(X[t,i] - X[t,j])
            D[i, j] = s
            D[j, i] = s
    return D

# Sanity check on window 1
X1, tick1 = formation_paths(windows[0])
t0 = time.time()
D_man1 = manhattan_matrix(X1)
print(f'Window 1: {X1.shape[1]} stocks, {X1.shape[0]} formation days, '
      f'Manhattan matrix {D_man1.shape} in {time.time()-t0:.2f}s')
print(f'Symmetric: {np.allclose(D_man1, D_man1.T)}, zero diagonal: {np.all(np.diag(D_man1) == 0)}')
i, j = np.unravel_index(np.argmin(D_man1 + np.eye(len(tick1)) * 1e9), D_man1.shape)
print(f'Closest pair in window 1: {tick1[1]}-{tick1[j]} (distance {D_man1[i,j]:.2f})')

Window 1: 221 stocks, 252 formation days, Manhattan matrix (221, 221) in 1.50s
Symmetric: True, zero diagonal: True
Closest pair in window 1: ABT-OMC (distance 10.25)


In [18]:
# DTW distance: same absolute-difference cost as Manhattan, but any warping path is allowed
@njit(cache=True)
def dtw_abs(x, y):
    n, m = len(x), len(y)
    prev = np.full(m, np.inf)
    for i in range(n):
        cur = np.full(m, np.inf)
        for j in range(m):
            if i == 0 and j == 0:
                cur[0] = abs(x[0] - y[0])
                continue
            best = np.inf
            if i > 0 and prev[j] < best:
                best = prev[j]
            if j > 0 and cur[j - 1] < best:
                best = cur[j - 1]
            if i > 0 and j > 0 and prev[j - 1] < best:
                best = prev[j - 1]
            cur[j] = abs(x[i] - y[j]) + best
        prev = cur
    return prev[m - 1]

@njit(parallel=True, cache=True)
def dtw_matrix(X):
    T, n = X.shape
    D = np.zeros((n,n))
    for i in prange(n):
        for j in range(i + 1, n):
            d = dtw_abs(X[:, i], X[:, j])
            D[i, j] = d
            D[j, i] = d
    return D

# Sanity check on window 1
t0 = time.time()
D_dtw1 = dtw_matrix(X1)
print(f'DTW  matrix:{D_dtw1.shape} in {time.time()-t0:.2f}s')
print(f'Symmetric: {np.allclose(D_dtw1, D_dtw1.T)}, zero diagonal: {np.all(np.diag(D_dtw1) == 0)}')
print(f'DTW never exceeds Manhattan: {np.all(D_dtw1 <= D_man1 + 1e-9)}')
iu = np.triu_indices(len(tick1), k=1)
print(f'Average DTW / Manhattan ratio: {np.mean(D_dtw1[iu] / D_man1[iu]):.3f}')
i, j = np.unravel_index(np.argmin(D_dtw1 + np.eye(len(tick1)) * 1e9), D_dtw1.shape)
print(f'Closest pair in window 1: {tick1[i]}-{tick1[j]} (distance: {D_dtw1[i, j]:.2f})')

DTW  matrix:(221, 221) in 6.98s
Symmetric: True, zero diagonal: True
DTW never exceeds Manhattan: True
Average DTW / Manhattan ratio: 0.651
Closest pair in window 1: ECL-ITT (distance: 4.18)


In [19]:
# OCP distance: DTW restricted to causal paths (j >= i), computed in both directions, smaller kept 
@njit(cache=True)
def ocp_one_way(x, y):
    n, m = len(x), len(y)
    prev = np.full(m, np.inf)
    for i in range(n):
        cur = np.full(m, np.inf)
        for j in range(i, m):              # OCP: only j >= i (causality constraint)
            if i == 0 and j == 0:
                cur[0] = abs(x[0] - y[0])
                continue
            best = np.inf
            if i > 0 and prev[j] < best:
                best = prev[j]
            if j > 0 and cur[j - 1] < best:
                best = cur[j - 1]
            if i > 0 and j > 0 and prev[j - 1] < best:
                best = prev[j - 1]
            cur[j] = abs(x[i] - y[j]) + best
        prev = cur
    return prev[m - 1]

@njit(parallel=True, cache=True)
def ocp_matrix(X):
    T, n = X.shape
    D = np.zeros((n, n))
    for i in prange(n):
        for j in range(i + 1, n):
            d = min(ocp_one_way(X[:, i], X[:, j]), ocp_one_way(X[:, j], X[:, i]))
            D[i, j] = d
            D[j, i] = d
    return D

# Sanity check on window 1
t0 = time.time()
D_ocp1 = ocp_matrix(X1)
print(f'OCP matrix {D_ocp1.shape} in {time.time()-t0:.2f}s')
print(f'Symmetric: {np.allclose(D_ocp1, D_ocp1.T)}, zero diagonal: {np.all(np.diag(D_ocp1) == 0)}')
print(f'DTW <= OCP: {np.all(D_dtw1 <= D_ocp1 + 1e-9)}')
print(f'OCP <= Manhattan: {np.all(D_ocp1 <= D_man1 + 1e-9)}')
print(f'Average OCP/ Manhattan ratio: {np.mean(D_ocp1[iu] / D_man1[iu]):.3f}')
i, j = np.unravel_index(np.argmin(D_ocp1 + np.eye(len(tick1)) * 1e9), D_ocp1.shape)
print(f'Closest pair in window 1: {tick1[i]}-{tick1[j]} (distance {D_ocp1[i, j]:.2f})')

OCP matrix (221, 221) in 8.14s
Symmetric: True, zero diagonal: True
DTW <= OCP: True
OCP <= Manhattan: True
Average OCP/ Manhattan ratio: 0.677
Closest pair in window 1: ECL-ITT (distance 4.53)


In [ ]:
# Ward clustering with k chosen per window by majority vote of three indices (silhouette breaks ties)
def cluster_and_choose_K(D, X):
    link = linkage(squareform(D, checks=False), method='ward')
    feats = X.T
    rows, labels = [], {}
    for k in K_OPTIONS:
        lab = fcluster(link, t=k, criterion='maxclust')
        labels[k] = lab
        rows.append({'k': k,
                     'silhouette': silhouette_score(D, lab, metric='precomputed'),
                     'calinski-harabasz': calinski_harabasz_score(feats, lab),
                     'davies_bouldin': davies_bouldin_score(feats, lab)})
        scores = pd.DataFrame(rows).set_index('k')
        votes = {'silhouette': int[(scores['silhouette'].idmax()),
                 'calinski-harabasz': int(scores['calinski-harabasz'].idmax()),
                 'davies_bouldin': int(scores['davies_bouldin'].idmin())}
        counts = pd.Series(list(votes.values())).value_counts()
        k = int(counts.index[0]) if counts.iloc[0] >=  2 else votes['silhouette']
        agree = int(counts.iloc[0])
        return labels[k], k, agree, votes, scores
